In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Path:
# /content/drive/MyDrive/ablation_outputs/system_1_bm25/S1_bm25_error_bank_rev4_results.csv
# /content/drive/MyDrive/ablation_outputs/system_1_bm25/S1_bm25_error_bank_rev5_results.csv
# /content/drive/MyDrive/ablation_outputs/system_2_dense/S2_dense_error_bank_rev4_results.csv
# /content/drive/MyDrive/ablation_outputs/system_2_dense/S2_dense_error_bank_rev5_results.csv
# /content/drive/MyDrive/ablation_outputs/system_3_rewrite_only/S3_qur_replace_error_bank_rev4_results.csv
# /content/drive/MyDrive/ablation_outputs/system_3_rewrite_only/S3_qur_replace_error_bank_rev5_results.csv
# /content/drive/MyDrive/ablation_outputs/system_4_qur_rrf/S4_qur_rrf_error_bank_rev4_results.csv
# /content/drive/MyDrive/ablation_outputs/system_4_qur_rrf/S4_qur_rrf_error_bank_rev5_results.csv
# /content/drive/MyDrive/ablation_outputs/system_5_hybrid_rrf/S5_hybrid_rrf_error_bank_rev4_results.csv
# /content/drive/MyDrive/ablation_outputs/system_5_hybrid_rrf/S5_hybrid_rrf_error_bank_rev5_results.csv
# /content/drive/MyDrive/ablation_outputs/system_6_hybrid_rerank/S6_hybrid_rerank_error_bank_rev4_results.csv
# /content/drive/MyDrive/ablation_outputs/system_6_hybrid_rerank/S6_hybrid_rerank_error_bank_rev5_results.csv
# /content/drive/MyDrive/ablation_outputs/system_7_compliance_gpt/S7_compliance_gpt_error_bank_rev4_results.csv
# /content/drive/MyDrive/ablation_outputs/system_7_compliance_gpt/S7_compliance_gpt_error_bank_rev5_results.csv

In [6]:
import pandas as pd
import numpy as np
import os # Import os to check if directory exists

# --- 1. Define Helper Function for RR@10 ---
def calculate_rr_at_10(rank):
    """Calculates the Reciprocal Rank at 10."""
    try:
        rank = pd.to_numeric(rank, errors='coerce')
        if pd.isna(rank) or rank == 0:
            return 0.0
        if 1.0 <= rank <= 10.0:
            return 1.0 / rank
        else:
            return 0.0
    except Exception:
        return 0.0

print("--- Reproducibility Script for Task Started ---")
print("Helper function calculate_rr_at_10() defined.\n")

# --- 2. Define All File Paths ---

# Base directory
base_dir = "/content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/"

# Path to the master error bank file
master_file = "/content/drive/MyDrive/ComplianceGPT_v2/data/error_bank/error_bank_v1.csv"

## NOTE: Including all 5 systems.
system_files = {
    'S1_MRR': [
        base_dir + "system_1_bm25/S1_bm25_error_bank_rev4_results.csv",
        base_dir + "system_1_bm25/S1_bm25_error_bank_rev5_results.csv"
    ],
    'S2_MRR': [
        base_dir + "system_2_dense/S2_dense_error_bank_rev4_results.csv",
        base_dir + "system_2_dense/S2_dense_error_bank_rev5_results.csv"
    ],
    'S5_MRR': [
        base_dir + "system_5_hybrid_rrf/S5_hybrid_rrf_error_bank_rev4_results.csv",
        base_dir + "system_5_hybrid_rrf/S5_hybrid_rrf_error_bank_rev5_results.csv"
    ],
    'S6_MRR': [
        base_dir + "system_6_hybrid_rerank/S6_hybrid_rerank_error_bank_rev4_results.csv",
        base_dir + "system_6_hybrid_rerank/S6_hybrid_rerank_error_bank_rev5_results.csv"
    ],
    'S7_MRR': [
        base_dir + "system_7_compliance_gpt/S7_compliance_gpt_error_bank_rev4_results.csv",
        base_dir + "system_7_compliance_gpt/S7_compliance_gpt_error_bank_rev5_results.csv"
    ]
}

print("Defined all system file paths.\n")

# --- 3. Load Master File ---
try:
    master_df = pd.read_csv(master_file)
    # We need 'question' (as the key) and 'failure_category' (for the pivot)
    master_df = master_df[['question_id', 'question', 'failure_category']]
    print(f"Loaded master file: '{master_file}' ({len(master_df)} queries)")
    print(f"Categories found:\n{master_df['failure_category'].value_counts().to_string()}\n")
    merged_data_df = master_df.copy()

except Exception as e:
    print(f"CRITICAL ERROR: Could not load master file '{master_file}'.")
    print(f"Error: {e}")
    # Stop execution if master file fails
    raise

# --- 4. Process Each System's Results ---
print("--- Starting System MRR Calculation Loop ---")
for system_name, files in system_files.items():
    try:
        # Check if files exist before trying to read
        existing_files = [f for f in files if os.path.exists(f)]
        if not existing_files:
            print(f"WARNING: No files found for {system_name} at paths: {files}. Skipping.")
            continue

        df_list = [pd.read_csv(f) for f in existing_files]
        system_df = pd.concat(df_list, ignore_index=True)

        if 'rank' not in system_df.columns or 'question' not in system_df.columns:
            print(f"WARNING: 'rank' or 'question' column not found for {system_name}. Skipping.")
            continue

        system_df[system_name] = system_df['rank'].apply(calculate_rr_at_10)
        # system_rr_df = system_df.groupby('question')[[system_name]].max().reset_index()
        system_rr_df = system_df.groupby("question_id")[[system_name]].max().reset_index()
        merged_data_df = pd.merge(merged_data_df, system_rr_df, on="question_id", how="left")


        print(f"[SUCCESS] Loaded and calculated MRR for {system_name}.")

    except Exception as e:
        print(f"[ERROR] Processing files for {system_name}: {e}")
print("--- System MRR Calculation Loop Finished ---\n")

# --- 5. Create Pivot Table and Save Artifacts ---
mrr_columns = [col for col in merged_data_df.columns if col.startswith('S') and '_MRR' in col]

if mrr_columns:
    merged_data_df[mrr_columns] = merged_data_df[mrr_columns].fillna(0.0)
    pivot_table = merged_data_df.groupby('failure_category')[mrr_columns].mean()

    # Re-order columns to S1, S2, S3, S4, S5, S6, S7 for readability
    ordered_cols = sorted(mrr_columns)
    pivot_table = pivot_table[ordered_cols]

    print("\n--- Task: Error Bank by Failure Mode (MRR@10) ---")
    print(pivot_table.to_markdown(floatfmt=".4f"))

    # --- EDITED SAVE LOGIC ---

    # Define the output directory
    output_dir = "/content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/error_analysis/generated_reports"

    # Create the directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n--- Saving output files to: {output_dir} ---")

    # Define file paths
    csv_pivot_path = os.path.join(output_dir, 'pivot_table.csv')
    md_pivot_path = os.path.join(output_dir, 'pivot_table.md')
    merged_data_path = os.path.join(output_dir, 'merged_results_data.csv')

    # Save the pivot table to CSV (with 4-digit precision)
    pivot_table.to_csv(csv_pivot_path, float_format='%.4f')
    print(f"[SUCCESS] Saved pivot table CSV to '{csv_pivot_path}'.")

    # Save the pivot table to Markdown (with 4-digit precision)
    pivot_table.to_markdown(md_pivot_path, floatfmt=".4f")
    print(f"[SUCCESS] Saved pivot table Markdown to '{md_pivot_path}'.")

    # Save the merged results data (with 4-digit precision)
    merged_data_df.to_csv(merged_data_path, index=False, float_format='%.4f')
    print(f"[SUCCESS] Saved merged data CSV to '{merged_data_path}'.")

    # --- END OF EDITED SAVE LOGIC ---

else:
    print("\n[ERROR] No MRR data was successfully processed. Could not generate pivot table.")

print("\n--- Reproducibility Script Finished ---")

--- Reproducibility Script for Task Started ---
Helper function calculate_rr_at_10() defined.

Defined all system file paths.

Loaded master file: '/content/drive/MyDrive/ComplianceGPT_v2/data/error_bank/error_bank_v1.csv' (37 queries)
Categories found:
Series([], )

--- Starting System MRR Calculation Loop ---
[SUCCESS] Loaded and calculated MRR for S1_MRR.
[SUCCESS] Loaded and calculated MRR for S2_MRR.
[SUCCESS] Loaded and calculated MRR for S5_MRR.
[SUCCESS] Loaded and calculated MRR for S6_MRR.
[SUCCESS] Loaded and calculated MRR for S7_MRR.
--- System MRR Calculation Loop Finished ---


--- Task: Error Bank by Failure Mode (MRR@10) ---
| failure_category   | S1_MRR   | S2_MRR   | S5_MRR   | S6_MRR   | S7_MRR   |
|--------------------|----------|----------|----------|----------|----------|

--- Saving output files to: /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/error_analysis/generated_reports ---
[SUCCESS] Saved pivot table CSV to '/content/drive/MyDriv